# HRS Silver CDM DML Workflow

---

## 1. Document Information

| Property          | Value                                                |
| ----------------- | ---------------------------------------------------- |
| Document Name     | HRS Silver CDM DML Workflow                          |
| Version           | 1.0                                                  |
| Author            | Perez                                                |
| AI Assistant      | ChatGPT                                              |
| Last Updated      | 2026-09-09                                           |
| Target Platform   | Databricks                                           |
| Compute           | Serverless                                           |
| Runtime           | client.5.12                                          |
| SQL Dialect       | Databricks SQL / Spark SQL                           |
| Storage Format    | Delta                                                |
| Target Data Layer | Silver CDM                                           |
| Workflow Type     | DML Development, Generation, Testing, and Validation |

---

# 2. Workflow Objective

The objective of this workflow is to provide a standardized process for designing, generating, testing, validating, and executing SQL DML used to transform RAND HRS source data and load it into an existing HRS Silver CDM target table.

The workflow begins only after the corresponding target Delta table has been successfully created and validated through the HRS Silver CDM DDL Workflow.

The DML workflow ensures that:

1. The source RAND HRS variables are properly identified.
2. Source-to-target mappings are explicitly defined.
3. Respondent and wave surrogate keys are correctly resolved.
4. The wide RAND HRS longitudinal structure is transformed into the required Silver CDM structure.
5. Data types and transformations are explicitly defined.
6. RAND HRS missing-value conventions are handled according to the specification.
7. Audit columns are populated correctly.
8. The target business grain is preserved.
9. Duplicate observations are identified and prevented.
10. The generated DML is tested before production use.
11. Source-to-target results can be reconciled and validated.

---

# 3. DML Workflow Prerequisite

The DML workflow requires a completed and validated DDL workflow.

The required sequence is:

```text id="2m7r2e"
                 DDL WORKFLOW
                      │
                      ▼
             Target Requirements
                      │
                      ▼
              DDL Specification
                      │
                      ▼
               Generate DDL
                      │
                      ▼
              Create Delta Table
                      │
                      ▼
              Validate Structure
                      │
                      ▼
                 DDL APPROVED
                      │
                      ▼
              DML WORKFLOW
```

The DML workflow must not create or redefine the target table.

---

# 4. DML Workflow Overview

The complete DML process is:

```text id="k2q3x5"
              VALIDATED SILVER TABLE
                       │
                       ▼
             1. Requirements Analysis
                       │
                       ▼
             2. DML Specification
                       │
                       ▼
             3. Source-to-Target Mapping
                       │
                       ▼
             4. Transformation Design
                       │
                       ▼
             5. DML SQL Generation
                       │
                       ▼
             6. DML Code Review
                       │
                       ▼
             7. Test / Execute DML
                       │
                       ▼
             8. Data Validation
                       │
                       ▼
             9. Reconciliation Testing
                       │
                       ▼
                DML APPROVED
                       │
                       ▼
                Production Load
```

---

# 5. Step 1 — Confirm DDL Completion

Before beginning DML development, confirm that the target table has passed the DDL validation process.

Verify:

* Target table exists.
* Correct catalog.
* Correct schema.
* Correct table name.
* Delta format.
* Managed table.
* Required columns exist.
* Correct data types.
* Correct nullability.
* Identity column exists.
* Primary key exists.
* Respondent foreign key exists.
* Wave foreign key exists.
* Required comments exist.

### Gate 1 — DDL Approval

The DML workflow must not proceed until the target DDL has been approved.

---

# 6. Step 2 — Requirements Analysis

Document the data-loading requirements for the subject area.

## 6.1 Identify the Source

Document the RAND HRS source table.

Example:

```text id="7b8yr1"
dev_catalog.brz_raw_hrs.randhrs1992_2022v1
```

Confirm:

* Source catalog
* Source schema
* Source table
* Source description
* Source version

---

## 6.2 Identify the Target

Document the existing Silver CDM target table.

The target table should correspond to the approved DDL specification.

Example:

```text id="ax5j8k"
dev_catalog.slv_cdm_hrs.<TARGET_TABLE_NAME>
```

---

## 6.3 Define the Target Grain

The target grain must be explicitly documented.

For the HRS Silver CDM architecture, the default grain is:

> One row per respondent per survey wave.

The logical business key is:

```text id="d9j2vf"
respondent_id + wave_id
```

The source natural identifiers are:

```text id="i5m6w4"
HHIDPN + wave_number
```

---

# 7. Step 3 — Define Natural Identifier Resolution

The DML must use natural identifiers to locate system-generated surrogate keys.

## 7.1 Respondent Lookup

`HHIDPN` is the natural respondent identifier.

The DML resolves:

```text id="e3h1yr"
Source HHIDPN
      │
      ▼
hub_respondent.HHIDPN
      │
      ▼
hub_respondent.respondent_id
      │
      ▼
Target.respondent_id
```

The DML must not generate `respondent_id`.

---

## 7.2 Wave Lookup

`wave_number` is the natural wave identifier.

The DML resolves:

```text id="7v4h0x"
Source wave_number
      │
      ▼
dim_wave.wave_number
      │
      ▼
dim_wave.wave_id
      │
      ▼
Target.wave_id
```

The DML must not generate `wave_id`.

---

# 8. Step 4 — Create the DML Specification

Create a subject-area DML specification using:

`HRS_DML_Master_Template.ipynb`

The DML specification should define:

* Source table
* Target table
* Target grain
* Natural identifiers
* Parent-table lookups
* Source variables
* Target columns
* Wave applicability
* Wave-specific variables
* Wave-invariant variables
* Transformations
* Data-type conversions
* Missing-value rules
* NULL handling
* Filtering rules
* Audit-column rules
* Duplicate handling
* Load pattern
* Validation requirements

The DML specification must reference the corresponding DDL specification.

It should not duplicate the physical target-table definition unless required to explain a transformation.

---

# 9. Step 5 — Source-to-Target Mapping

Create the source-to-target mapping matrix.

The matrix is the authoritative definition of how source variables populate target columns.

Recommended structure:

| Wave     | Source Variable     | Variable Label | RAND Type     | Target Column     | Databricks Type | Transformation | Nullable | Missing-Value Rule |
| -------- | ------------------- | -------------- | ------------- | ----------------- | --------------- | -------------- | -------- | ------------------ |
| `<wave>` | `<source_variable>` | `<label>`      | `<RAND type>` | `<target_column>` | `<type>`        | `<rule>`       | Yes/No   | `<rule>`           |

Every target business column populated by the DML should have an explicitly defined source mapping or transformation rule.

---

# 10. Step 6 — Identify Wave-Specific Variables

RAND HRS longitudinal data contains many wave-specific variables.

For example:

```text id="p6c5ah"
R1AGEY_E
R2AGEY_E
R3AGEY_E
...
R16AGEY_E
```

These variables represent different waves of the same logical target attribute.

The DML must transform these variables into the target respondent-wave structure.

Example:

```text id="85ol2z"
Source                  Target

R1AGEY_E     ────────►  Wave 1 / agey_e
R2AGEY_E     ────────►  Wave 2 / agey_e
R3AGEY_E     ────────►  Wave 3 / agey_e
...
R16AGEY_E    ────────►  Wave 16 / agey_e
```

---

# 11. Step 7 — Identify Wave-Invariant Variables

The DML specification must identify variables that occur once in the source rather than once for every wave.

Examples may include:

```text id="4z7t4b"
RARACEM
RAHISPAN
RAEDYRS
RARELIG
RAVETRN
```

The specification must explicitly define how these variables are associated with respondent-wave observations.

The DML generator must not assume the appropriate behavior when the specification does not define it.

---

# 12. Step 8 — Design the Wide-to-Long Transformation

The RAND HRS source is generally organized in a wide longitudinal format.

The Silver CDM target is organized around respondent-wave observations.

Therefore, the DML may require a wide-to-long transformation.

Conceptually:

```text id="q2o5by"
                 RAND HRS SOURCE
                       │
                       ▼
              Wide Longitudinal Data
                       │
                       ▼
                  UNPIVOT
                       │
                       ▼
               Wave-Level Rows
                       │
                       ▼
              Respondent + Wave
                       │
                       ▼
                 SILVER CDM
```

For example:

```text id="3b5g0t"
HHIDPN   R1AGEY_E   R2AGEY_E   R3AGEY_E
12345       65         67         69
```

may become conceptually:

```text id="n8o0h4"
HHIDPN   wave_number   agey_e
12345         1           65
12345         2           67
12345         3           69
```

The exact SQL transformation must be determined by the subject-area DML specification.

---

# 13. Step 9 — Define Data Transformations

Each transformation must be explicitly documented.

Possible transformations include:

* Direct mapping
* CAST
* Type conversion
* Code translation
* NULL conversion
* Missing-value handling
* Wide-to-long transformation
* Unpivoting
* Conditional transformation
* Derived value
* Filtering

Example:

```text id="e8y7qx"
Source: R1AGEY_E
Target: agey_e
Transformation: CAST to DECIMAL(10,2)
```

The DML generator must not invent transformation logic.

---

# 14. Step 10 — Define RAND HRS Missing-Value Rules

RAND HRS data may contain special values representing different types of missing or non-substantive responses.

The DML specification must explicitly define how these values are handled.

For each applicable variable, document:

* Source missing-value code
* Meaning
* Target representation
* Transformation rule

Example:

```text id="v4af7c"
Source Code
    │
    ▼
RAND HRS Missing-Value Rule
    │
    ▼
Target Value
```

Missing-value handling must be explicitly defined in the DML specification.

The generator must not automatically convert values without a documented rule.

---

# 15. Step 11 — Define Audit Columns

The DML must populate required audit columns according to the target DDL specification.

Typical audit columns include:

```text id="0q2r9f"
create_date
update_date
active
```

The DML specification must define how each audit column is populated.

For example:

```text id="0yr8aw"
create_date → Current load date
update_date → Current load date
active      → TRUE
```

The exact rule must be specified.

---

# 16. Step 12 — Define Load Pattern

The DML specification must identify the required load pattern.

Examples include:

* Insert Only
* Full Refresh
* Incremental Insert
* Incremental Update
* Slowly Changing Dimension processing

For the applicable HRS subject-area table, the load pattern must be explicitly documented.

The generated DML must follow the specified load pattern.

---

# 17. Step 13 — Generate DML SQL

Once the DML specification has been approved, provide it to the AI Assistant using:

`HRS_DML_Master_Template.ipynb`

The AI Assistant generates the DML SQL from the approved specification.

The generated SQL should include the necessary transformation and load logic.

A typical structure may be:

```text id="q0u3uw"
WITH
source_data AS (...),
wave_data AS (...),
respondent_lookup AS (...),
wave_lookup AS (...),
transformed_data AS (...)
INSERT INTO target_table
(
    respondent_id,
    wave_id,
    ...
)
SELECT
    respondent_id,
    wave_id,
    ...
FROM transformed_data;
```

The actual structure depends on the DML specification.

---

# 18. Step 14 — DML Code Review

Review the generated SQL against the approved DML specification.

Verify:

* Source table
* Target table
* Source variables
* Target columns
* Respondent lookup
* Wave lookup
* Wave transformations
* Unpivot logic
* Data-type conversions
* Missing-value handling
* NULL handling
* Audit columns
* Filters
* Load pattern
* Duplicate handling

The generated SQL must not contain undocumented business rules.

---

# 19. Step 15 — DML Unit Testing

Before loading the production target table, test the transformation logic.

Testing should initially be performed using controlled queries or a test target where appropriate.

## 19.1 Source Record Testing

Confirm that the expected source population is being selected.

Test:

* Number of source respondents
* Number of source waves
* Number of source observations

---

## 19.2 Respondent Lookup Testing

Verify that `HHIDPN` correctly resolves to `respondent_id`.

Test for:

* Successful matches
* Unmatched HHIDPN values
* Duplicate parent identifiers

---

## 19.3 Wave Lookup Testing

Verify that `wave_number` correctly resolves to `wave_id`.

Test for:

* Successful matches
* Unmatched wave numbers
* Duplicate wave identifiers

---

## 19.4 Transformation Testing

Validate:

* Wave-specific mappings
* Wave-invariant mappings
* Unpivot logic
* Data-type conversions
* Derived values
* Missing-value rules

---

## 19.5 Business-Grain Testing

Confirm that the transformed data contains no duplicate:

```text id="3xy0yw"
respondent_id + wave_id
```

Example validation concept:

```sql
SELECT
    respondent_id,
    wave_id,
    COUNT(*) AS row_count
FROM transformed_data
GROUP BY
    respondent_id,
    wave_id
HAVING COUNT(*) > 1;
```

The expected result is zero rows.

---

# 20. Step 16 — Execute the DML

After unit testing and code review, execute the approved DML against the target table.

The execution process is:

```text id="3w3kn9"
Approved DML
     │
     ▼
Execute in Databricks
     │
     ▼
Transform Source Data
     │
     ▼
Resolve Parent Keys
     │
     ▼
Load Target Table
```

---

# 21. Step 17 — Validate Loaded Data

After execution, validate the target table.

## 21.1 Row Count Validation

Compare expected and actual target row counts.

```text id="v0r5ul"
Expected Rows
     │
     ▼
Actual Rows
     │
     ▼
Reconcile Difference
```

---

## 21.2 Respondent Validation

Confirm that loaded respondents have valid:

```text id="i2v76k"
respondent_id
```

and that each value exists in:

```text id="y6o2bb"
hub_respondent.respondent_id
```

---

## 21.3 Wave Validation

Confirm that loaded observations have valid:

```text id="j9p2te"
wave_id
```

and that each value exists in:

```text id="8z6i8h"
dim_wave.wave_id
```

---

## 21.4 Business-Grain Validation

Confirm that there are no duplicate:

```text id="v3q5xu"
respondent_id + wave_id
```

---

## 21.5 Attribute Validation

Validate selected source and target values.

For example:

```text id="8trq6p"
RAND HRS Source
      │
      ▼
Transformation
      │
      ▼
Silver CDM Target
```

Sample records should be manually reconciled to confirm that the transformation produced the expected results.

---

# 22. Step 18 — Source-to-Target Reconciliation

Perform reconciliation between the source and target.

At a minimum, evaluate:

| Validation                     | Expected        |
| ------------------------------ | --------------- |
| Source respondents             | Reconciles      |
| Target respondents             | Reconciles      |
| Source waves                   | Reconciles      |
| Target waves                   | Reconciles      |
| Source observations            | Reconciles      |
| Target observations            | Reconciles      |
| Unmatched respondents          | 0 or documented |
| Unmatched waves                | 0 or documented |
| Duplicate respondent-wave rows | 0               |
| Unexpected NULL foreign keys   | 0               |
| Transformation exceptions      | 0 or documented |

Any differences must be investigated and documented.

---

# 23. DML Validation Gate

The DML workflow is complete only when all required tests have passed.

### DML Completion Criteria

| Validation                                | Required |
| ----------------------------------------- | :------: |
| DDL approved                              |     ✓    |
| DML specification approved                |     ✓    |
| Source mappings approved                  |     ✓    |
| Natural identifiers defined               |     ✓    |
| Respondent lookup tested                  |     ✓    |
| Wave lookup tested                        |     ✓    |
| Wave transformation tested                |     ✓    |
| Missing-value rules tested                |     ✓    |
| Data-type conversions tested              |     ✓    |
| Audit columns tested                      |     ✓    |
| DML code reviewed                         |     ✓    |
| DML executed successfully                 |     ✓    |
| Target row counts validated               |     ✓    |
| Respondent keys validated                 |     ✓    |
| Wave keys validated                       |     ✓    |
| Duplicate respondent-wave rows            |     0    |
| Source-to-target reconciliation completed |     ✓    |
| Exceptions documented                     |     ✓    |

---

# 24. DML Approval

Once validation is complete, the subject-area DML is approved.

The final state is:

```text id="k7m1hj"
             DML DEVELOPMENT
                    │
                    ▼
             DML SPECIFICATION
                    │
                    ▼
              DML GENERATED
                    │
                    ▼
               CODE REVIEW
                    │
                    ▼
              UNIT TESTING
                    │
                    ▼
             DML EXECUTION
                    │
                    ▼
            DATA VALIDATION
                    │
                    ▼
             RECONCILIATION
                    │
                    ▼
              DML APPROVED
                    │
                    ▼
             PRODUCTION LOAD
```

---

# 25. DML Workflow Handoff

The completed DML process produces a validated Silver CDM dataset.

The overall HRS Silver CDM development lifecycle is:

```text id="l3j7f4"
                 REQUIREMENTS
                      │
                      ▼
               DDL WORKFLOW
                      │
                      ▼
             TARGET TABLE CREATED
                      │
                      ▼
             TABLE VALIDATED
                      │
                      ▼
                DML WORKFLOW
                      │
                      ▼
             DATA TRANSFORMED
                      │
                      ▼
              DATA LOADED
                      │
                      ▼
             DATA VALIDATED
                      │
                      ▼
              CDM COMPLETE
```

---

# 26. Roles and Responsibilities

| Activity                     | User | AI Assistant | Databricks |
| ---------------------------- | :--: | :----------: | :--------: |
| Define business requirements |   ✓  |    Assist    |            |
| Identify source variables    |   ✓  |    Assist    |            |
| Define source mappings       |   ✓  |    Assist    |            |
| Define transformations       |   ✓  |    Assist    |            |
| Approve DML specification    |   ✓  |              |            |
| Generate DML                 |      |       ✓      |            |
| Review DML                   |   ✓  |    Assist    |            |
| Execute DML                  |   ✓  |    Assist    |      ✓     |
| Transform source data        |      |              |      ✓     |
| Load target table            |      |              |      ✓     |
| Validate results             |   ✓  |    Assist    |      ✓     |
| Approve final DML            |   ✓  |              |            |

The user remains responsible for business requirements, source mappings, transformation rules, and final approval.

The AI Assistant is responsible for translating the approved specification into SQL.

Databricks executes the SQL and performs the actual transformation and loading operations.

---

# 27. DML Design Principles

### Principle 1 — DDL Before DML

The target table must exist and be validated before DML development begins.

### Principle 2 — Specification Before SQL

The DML specification must be completed before generating the final DML.

### Principle 3 — Explicit Mappings

Every source-to-target mapping must be explicitly defined.

### Principle 4 — Natural Identifiers Resolve Surrogate Keys

`HHIDPN` is used to resolve `respondent_id`.

`wave_number` is used to resolve `wave_id`.

The DML must not generate these surrogate keys.

### Principle 5 — Preserve Business Grain

The default Silver CDM grain is one respondent per survey wave.

### Principle 6 — Explicit Transformations

Transformations must be documented rather than inferred.

### Principle 7 — Explicit Missing-Value Rules

RAND HRS missing-value handling must be explicitly specified.

### Principle 8 — Validate Before Approval

Successful SQL execution does not mean that the DML is correct.

The loaded data must be validated and reconciled.

### Principle 9 — No Invented Business Rules

The AI Assistant must not invent mappings, transformations, filters, or business rules that are not defined in the DML specification.

---

# 28. Final DML Workflow

The complete workflow is:

```text id="5qj3ne"
1. CONFIRM DDL COMPLETION
             │
             ▼
       Target Table Exists
             │
             ▼
       Target Structure Validated
             │
             ▼
2. REQUIREMENTS ANALYSIS
             │
             ▼
       Identify Source
             │
             ▼
       Identify Target
             │
             ▼
       Define Target Grain
             │
             ▼
3. DML SPECIFICATION
             │
             ▼
       Define Natural IDs
             │
             ▼
       Define Parent Lookups
             │
             ▼
       Define Source Mappings
             │
             ▼
       Define Transformations
             │
             ▼
       Define Missing-Value Rules
             │
             ▼
       Define Load Pattern
             │
             ▼
4. DML GENERATION
             │
             ▼
       AI Generates DML
             │
             ▼
5. DML CODE REVIEW
             │
             ▼
       Refine / Correct SQL
             │
             ▼
6. DML UNIT TESTING
             │
             ▼
       Test Lookups
             │
             ▼
       Test Transformations
             │
             ▼
       Test Business Grain
             │
             ▼
7. DML EXECUTION
             │
             ▼
       Transform Source
             │
             ▼
       Load Target
             │
             ▼
8. DATA VALIDATION
             │
             ▼
       Row Counts
             │
             ▼
       Keys
             │
             ▼
       Attributes
             │
             ▼
       Duplicate Testing
             │
             ▼
9. RECONCILIATION
             │
             ▼
       Source vs Target
             │
             ▼
       Exceptions Documented
             │
             ▼
         DML APPROVED
```

---

# 29. Workflow Completion

The HRS Silver CDM DML workflow is considered complete when:

1. The corresponding DDL has been approved.
2. The target Delta table exists.
3. The target table structure has been validated.
4. The DML requirements have been documented.
5. The DML specification has been approved.
6. Source-to-target mappings have been defined.
7. Natural identifier lookups have been defined and tested.
8. Wave transformations have been defined and tested.
9. Missing-value rules have been defined and tested.
10. The DML SQL has been generated.
11. The DML SQL has been reviewed.
12. Unit testing has passed.
13. The DML has executed successfully.
14. Target data has been validated.
15. Source-to-target reconciliation has been completed.
16. Exceptions have been documented.
17. The subject-area DML has been approved.

At this point, the HRS Silver CDM subject-area table contains validated data and is ready for downstream analytical use.
